# 28 · Relate a column from another table, checked against a pandas merge

## What this checks

**Relate a column from another table** brings a column from one table onto the matching rows of another. This check
covers release 0.15.0.

Snow's Table XII says which water company supplied each of 32 sub-districts. His Table VIII gives each sub-district's
population and cholera deaths, as 64 rows. Bringing the company onto those 64 rows is what lets the deaths be split by
supply. It is the first step of the chain behind the South London example's two-by-two table.

**No number is computed here, and that is the point.** The widget brings a *category* across. So this is a check on
the **matching**. It also checks two cases where Fieldwork is meant to behave differently from a plain join.

## How it is checked

We use **pandas**, Python's table library, by other authors. Its `merge` does a left join: it keeps every row of the
first table and adds the matching values from the second. It is what a practitioner would otherwise reach for. Where
it agrees, the agreement is worth having. Where it does not, the difference is the finding.

**Release 0.3.0 changed the widget in three ways**, so this check was rebuilt on 28 October 2026:

- **Folding.** Names are compared with punctuation and capitals ignored by default. This is called folding.
- **Refusing gaps.** A key the other table does not carry is refused until the practitioner says what to do with it.
- **Shares.** The result reports how much of each number column was kept.

We write the folding again from its declared rule. We never borrow Fieldwork's own, which would compare the widget
with itself.

The data hold eight cases, built from three input tables. One is new. The published sub-district boundary is a
second source for the same 32 names. It spells nine of them without their comma, for example "St George Camberwell"
for "St. George, Camberwell". That is the mismatch a practitioner will actually meet.

## What should happen

1. **The declared folding.** Normalise the text, trim spaces, and make it lower case. In the fuzzy mode, also turn
   every run of other characters into a single space, then trim again.
2. **Every row produced** matches a pandas left join on the folded name. We compare records as a set, so a different
   row order is not a disagreement.
3. **The row count is unchanged:** 64 in, 64 out. A step that multiplied rows would be a merge, and the widget says it
   is not one.
4. **The categories that landed**, in the order first seen, match the order a merge gives.
5. **A withheld name.** The same rows are dropped as a merge drops. The gap is reported by name, with the right count.
6. **An ambiguous name is refused.** The message must name the name and both values. Beside it we record how many
   rows a merge returns for the same input.
7. **An undeclared gap is refused.** A merge carries on and leaves empty values, which no total reveals.
8. **The share kept** of each number column, summed independently. From these follows an identity: a rate over the
   matched rows equals the rate over every row, times the ratio of the two shares kept.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_28_relate_table import run

report = run(Path.cwd() / "fixtures" / "relate-snow-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    print(f"\n{c['note']}")
    print(f"  compared {c['comparison']}, bringing {c['bring']}"
          f"{', declared ' + c['decision'] if c['decision'] else ''}")
    print(f"  keys matched   Fieldwork {c['fieldwork']['matchedKeys']:3d}   pandas {c['pandas']['matchedKeys']:3d}")
    print(f"  rows           {c['fieldwork']['rowCount']:12d}   {c['pandas']['rowCount']:9d}")
    print(f"  dropped        {c['fieldwork']['dropped']:12d}   {c['pandas']['dropped']:9d}")
    print(f"  kept           {(c['fieldwork']['kept'] or 0):12d}   {c['pandas']['kept']:9d}")
    for s in c["fieldwork"]["shares"]:
        mine = c["pandas"]["shares"][s["measure"]]
        pct = "-" if s["share"] is None else f"{s['share']*100:.2f}%"
        ours = "-" if mine["share"] is None else f"{mine['share']*100:.2f}%"
        print(f"  share {s['measure']:<20} {pct:>8}   {ours:>8}")
    for name, ok in c["checks"].items():
        print(f"    {'OK ' if ok else 'DIFF'} {name}")


## The result on record

First recorded on 27 October 2026, against release 0.3.0. Rerun on 26 November 2026, against release 0.15.0:
**Agrees**.

- **The relate the example runs:** 64 rows in, 64 out, none dropped. Three categories, in order. Identical to the
  merge, record for record.
- **One sub-district withheld:** 62 rows related and 2 dropped. The gap is reported as "St. Saviour, Southwark", 2
  rows. A merge drops the same two rows.
- **An ambiguous name, made up on purpose:** refused, with both company names in the message. The same input through
  a merge returns 66 rows from a 64-row table.

When the check was rebuilt on 28 October 2026, it moved no number on Snow's own two tables. They match 32 of 32 whether folded or not. So the new
default leaves the example's figures where they were.

Rebuilding also found a fault in the export. Release 0.3.0 had broken it outright. Two of the old cases withhold or
corrupt a name, and the widget now refuses an undeclared gap. Nothing in Fieldwork's checks runs the export, so
nothing had caught it.

Running the cell above repeats the check on the same data.

## The cases worth the check: two refusals

Fieldwork refuses twice where a merge carries on. Each refusal is the behaviour under check, not a limitation:

- **One name, two values.** The other table gives one name two different values. Fieldwork refuses outright. A merge
  multiplies the rows. Both companies then appear carrying the same population, and no total reveals it.
- **A name with nothing declared.** The other table does not carry the name, and the practitioner has said nothing
  about it. From release 0.3.0 this is refused too. A merge carries on and leaves an empty value.

The cell below shows what a merge does instead. That is what each refusal buys.

In [ ]:
for r in report["refusals"]:
    print(r["note"])
    if r.get("constructed"):
        print("  constructed:", r["constructed"])
    print("  Fieldwork refuses:")
    print("   ", r["fieldworkRefused"])
    print("  a merge instead:")
    print("   ", r["whatAMergeDoesInstead"])
    for name, ok in r["checks"].items():
        print(f"    {'OK ' if ok else 'DIFF'} {name}")
    print()


## The pair this check could not reach before

The earlier version of this notebook named a case it could not reach: "St. George, Camberwell" against "St George
Camberwell". That is the case a practitioner will actually meet, and neither its data nor its check reached it.

It is reachable now. The same two tables are compared both ways, so the two cases differ in one setting and nothing
else. Compared exactly, 23 of 32 names match. Folded, 32 of 32 match. That one setting moves eighteen rows.

**The shares are where it matters.** Without folding, the share kept is 68.24% of the population but 61.16% of the
deaths. The two columns do not fall in step. So leaving out the nine unmatched sub-districts moves the death rate
itself, not only how precise it is.

Fieldwork states an identity for this. A rate over the matched rows equals the rate over every row, times the ratio of
the shares kept. We recompute it from the tables and the shares separately: 3.109 × 0.8963 = 2.787 per 1,000,
against 2.787 observed. Fieldwork stated this arithmetic, but it had not been checked from outside before.

In [ ]:
pairs = [c for c in report["comparisons"] if c["bring"] == "supplier"]
for c in pairs:
    f, p = c["fieldwork"], c["pandas"]
    print(f"compared {c['comparison']:5}  keys {f['matchedKeys']:2}/32  rows {f['rowCount']:2}  dropped {f['dropped']:2}")
    for s in f["shares"]:
        print(f"    {s['measure']:<20} {s['related']:>9,.0f} of {s['all']:>9,.0f}"
              f"  = {(0 if s['share'] is None else s['share'])*100:6.2f}%")
print()
i = report["shareIdentity"]
print(i["claim"])
print(f"  rate over every offered row : {i['ratePerThousandAllRows']:.3f} per 1000")
print(f"  multiplier from the shares  : {i['multiplierFromShares']:.4f}")
print(f"  predicted                   : {i['ratePerThousandPredicted']:.3f} per 1000")
print(f"  observed                    : {i['ratePerThousandMatchedRows']:.3f} per 1000")
print(f"  holds                       : {i['holds']}")
print()
print(i["reading"])


## What this does not show

- **That folding is the right matching.** The cases show what each setting does to two published spellings of the
  same 32 sub-districts. They do not show that nine names differing by a comma name the same nine places. That is a
  judgement about the sources. A fold that joined two different places would be wrong in exactly the way this check
  cannot see.
- **That the practitioner's choice is the right one.** The check confirms that `drop` and `keep` give the totals each
  claims. Which a study should use is not a question a merge can answer.
- **That this is the right way to bring measures across.** That would add rows, not a column. It is deliberately not
  built, so nothing here speaks to it.
- **That the company is the right way to split the data.** Snow's third group, sub-districts supplied by both
  companies, holds 300,149 of the 486,936 people in the table. What to do with it is the practitioner's judgement, not
  an outcome of the join.

The cell below prints these limits as the check records them.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas  :", report["environment"]["pandas"])
print("runtime :", report["environment"]["runtime"])
